In [9]:
from typing import Any, Dict, List
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import json
import random
import sys, os
import math
import numpy as np
from uuid import UUID
from sentence_transformers import SentenceTransformer, util
from langchain_openai import AzureChatOpenAI

from deepeval.metrics import GEval, BaseMetric
from deepeval.test_case import LLMTestCaseParams, LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM

In [2]:
# Add parent directory to path for importing gepa and kontex
root_dir = Path(os.getcwd()).parent.resolve()
parent_dir = root_dir.parent

print(root_dir)
print(parent_dir)

/home/kunumi/projetos/kontex_gepa
/home/kunumi/projetos


In [3]:
# Add kontex src directory to path
kontex_src_dir = parent_dir /"kontex" / "src"
gepa_dir = parent_dir / "gepa" / "src" 

sys.path.insert(0, str(kontex_src_dir))
sys.path.insert(0, str(gepa_dir))

# 4. Configura o Banco de Dados na raiz para evitar confusão
os.environ['DATABASE_URL'] = f"sqlite:///{root_dir}/kontex_gepa_data.db"

print("\nCaminhos configurados. Tentando importar...")

try:
    import kontex
    import gepa
    print("✅ Sucesso! Módulos importados.")
except ModuleNotFoundError as e:
    print(f"❌ Erro: {e}")
    print(f"Verifique se existe um arquivo __init__.py em: {kontex_src_dir}/kontex/")



Caminhos configurados. Tentando importar...
✅ Sucesso! Módulos importados.


In [4]:
from dotenv import load_dotenv

env_path = root_dir / ".env"

if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅ Arquivo .env carregado de: {env_path}")
else:
    print("❌ Arquivo .env não encontrado na raiz!")

✅ Arquivo .env carregado de: /home/kunumi/projetos/kontex_gepa/.env


In [5]:
from kontex.logging import logger
from kontex.database import db
from kontex.knowledge import CollectedKnowledge
from kontex.simulation.edd.general_knowledge import FullKnowledge, DomainKnowledge, PredefinedKnowledge
from kontex.llm.scheduler import LLMScheduler
from kontex.llm.agents import DummyAgent
from kontex.settings import settings
from kontex.specialist import Specialist
from kontex.simulation.edd.simulation import edd_simulation
from kontex.simulation.edd.edd_run_params import EDDRunConfig
from kontex.orquestration import ConversationalWrapper
from kontex.llm.agents.hotpotqa_agents import (
    hotpotqa_description_building_role,
    hotpotqa_questioning_role,
    hotpotqa_self_critique_role,
    hotpotqa_subject_change_role,
)

from gepa import GEPAOptimizer, GEPAConfig
from gepa.core.system import CompoundAISystem, LanguageModule, SequentialFlow, IOSchema
from gepa.evaluation.base import SimpleEvaluator
from gepa.evaluation.metrics import ExactMatch, F1Score
from gepa.inference.factory import InferenceFactory
from gepa.config import InferenceConfig, OptimizationConfig, DatabaseConfig, ObservabilityConfig
from gepa.evaluation.base import Evaluator, EvaluationResult, SimpleEvaluator, SimpleFeedbackEvaluator
from gepa.evaluation.metrics import Metric

2026-03-20 15:24:28.470 | INFO     | kontex.database.database:__init__:40 - Initializing database connection to sqlite:////home/kunumi/projetos/kontex_gepa/kontex_gepa_data.db


/home/kunumi/projetos/kontex/src/kontex/llm


In [ ]:
def run_conversation_simulation(
    run_id: UUID,
    simulated_users: dict[str, Specialist],
    full_knowledge: FullKnowledge,
    prompts: dict[str, str] | None = None,
    seed: int = None,
    change_roles: bool | bool = False,
    new_roles: dict[str, str] | None = None,
    external_question: str | None = None
    ) -> dict[str, str]:

    rng = random.Random(seed)

    # TODO verificar como iremos lidar com múltiplas tabelas no futuro (se o agente tenta encontrar tudo de uma vez ou explora uma tabela por vez)
    descriptions = {}
    for table_name, table_knowledge in full_knowledge.domains.items():
        table_columns = list(table_knowledge.facts.keys())
        initial_description = f"Table: {table_name}\nColumns: {table_columns}"
        table = CollectedKnowledge(table_name, initial_description)

        scheduler = LLMScheduler(
                maxhist=0, new_roles=new_roles
                )  if change_roles else LLMScheduler(maxhist=0)  # Only use the most recent messages

        conversational_wrapper = ConversationalWrapper(
            scheduler,
            prompts,
            simulated_users,
            run_id,
        )
        initial_user = rng.choice(list(simulated_users.keys()))
        description, final_critique_score = conversational_wrapper.run_conversation(
            table,
            initial_user,
            min_description_quality=3,
            max_single_conversation=1,
            max_conversation_depth=10,  # Limit the conversation depth to avoid long runtimes during testing
            external_question=external_question,
            external_data=True,
            skip_hi_n_bye=True
        )

        logger.info(f"Final Table Description:\n{description}")
        logger.info(
            f"\n-------------\nOriginal Description: \n{table_knowledge.facts}"
        )
        logger.info(f"Final Critique Score: {final_critique_score}")
        descriptions[table_name] = description
    return descriptions, final_critique_score

class EnvConfig:
    """Configuration class to manage environment variables"""
    
    def __init__(self, env_file=".env"):
        # Load environment variables from .env file
        env_path = Path(env_file)
        
        if env_path.exists():
            load_dotenv(env_path)
            # print(load_dotenv(env_path))
            print(f"✓ Loaded environment variables from {env_file}")
        else:
            print(f"⚠ Warning: {env_file} file not found")
        
        # Load all configuration
        self.api_key = os.getenv("OPENAI_API_KEY")
        self.base_url = os.getenv("OPENAI_API_BASE")
        self.model = os.getenv("OPENAI_MODEL")

class AzureOpenAI(DeepEvalBaseLLM):
    def __init__(
        self,
        model
    ):
        self.model = model

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        return chat_model.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        res = await chat_model.ainvoke(prompt)
        return res.content

    def get_model_name(self):
        return "Custom Azure OpenAI Model"

class KontexFlowGeneralized():
    
    """A generalized control flow for Kontex that adapts to provided modules."""

    async def execute(
        self,
        modules: Dict[str, LanguageModule],
        input_data: Dict[str, Any],
        inference_client: Any
    ) -> Dict[str, Any]:
        """Execute modules in a dynamic manner based on available modules."""

        current_data = input_data.copy()

        new_roles = {
        ""
        "questioning": hotpotqa_questioning_role,
        "subject_change": hotpotqa_subject_change_role,
        "self_critique": hotpotqa_self_critique_role,
        "description_building": modules["description_building"].prompt,
        }

        prompts = {
            #"questioning_agent": '',  # Associado ao modulo questioning no CompoundAISystem do GEPA
            #"critique_agent": '', # Associado ao modulo self_critique no CompoundAISystem do GEPA
            'questioning': None,
            'self_critique': None,
            "description_building": None
            }

        # 2. Tratamento do run_id (Garantindo que seja o ID e não o objeto)
        raw_run_id = input_data.get("run_id", UUID(int=0))
        # Se o run_id for o objeto do SQLAlchemy do seu projeto anterior, pegamos apenas o atributo .id
        current_run_id = getattr(raw_run_id, 'id', raw_run_id)

        # 3. Execução da Simulação
        # Agora o dicionário 'prompts' contém tudo que foi definido no CompoundAISystem
        description, final_critique_score = run_conversation_simulation(
            run_id=current_run_id,
            prompts=prompts,
            simulated_users=input_data.get("users_with_knowledge", {}),
            full_knowledge=input_data.get("full_knowledge"),
            change_roles=True,
            new_roles = new_roles,
            seed=42,
            external_question = input_data.get("question")
        )

        # 4. Processamento de saída e métricas
        current_data['current_run_id'] = current_run_id
        
        # Garantir que o output seja numérico para o otimizador
        try:
            current_data['output'] = float(final_critique_score) if final_critique_score is not None else 0.0
        except (ValueError, TypeError):
            logger.warning(f"Could not convert score {final_critique_score} to float. Defaulting to 0.0")
            current_data['output'] = 0.0

        current_data['description'] = description
        current_data['output'] = final_critique_score
        logger.debug(f"Final critic score: {current_data['output']}")
        
        return current_data
    
class AverageDiffScore(Metric):
    """Average Difference Score Metric."""
    
    def __init__(self, name: str = "score"):
        super().__init__(name)
       
    
    def compute(self, predictions: List[Any], references: List[Any]) -> float:
        """Compute exact match score."""
        
        logger.debug(f"Predictions: {predictions}")
        logger.debug(f"References: {references}")

        scores = []
        for pred, ref in zip(predictions, references):
            diff = 10 - (ref - pred["output"])  # max score is 10
            scores.append(diff)

        logger.debug(f"Scores: {scores}")
        logger.debug(f"Mean score: {np.mean(np.array(scores))}")
        return np.mean(np.array(scores))
    
#: Critérios para o workflow HotPotQA (resposta a perguntas sobre múltiplos documentos)
GEVAL_CRITERIA_HOTPOTQA = {
    "factual_accuracy": (
        "Evaluate whether the answer produced by the questioning agent, based on facts "
        "collected from specialist agents, contains any fabricated, hallucinated, or "
        "incorrect information when compared to the expected answer. Penalize heavily "
        "for facts invented by the agent that are not supported by or contradict the "
        "expected answer."
    ),
    "completeness": (
        "Evaluate how thoroughly the questioning agent collected relevant facts from "
        "specialist agents to answer the original question. Assess whether the produced "
        "answer covers all key information present in the expected answer, and penalize "
        "for important facts or details that are missing."
    ),
}

class GEvalMetric(Metric):
    """
    Metric that makes use of different criteria
    """

    def __init__(self,  
                 name: str = "geval_metric", 
                 run_id: UUID = None,
                 factual_accuracy_criteria: str | None = None,
                 completeness_criteria: str | None = None,
                 ):
        super().__init__(name)
        self.name = name
        self.run_id = run_id  # Store run_id for database tracking

        self.factual_accuracy_criteria = (
            factual_accuracy_criteria
            or GEVAL_CRITERIA_HOTPOTQA["factual_accuracy"]
        )
        self.completeness_criteria = (
            completeness_criteria
            or GEVAL_CRITERIA_HOTPOTQA["completeness"]
        )

        config = EnvConfig(env_file = ".env")
        # Check for API key
        self.api_key = config.api_key
        self.model = config.model
        azure_endpoint = "https://azureopenai4k.openai.azure.com/"
        openai_api_version = "2025-01-01-preview"
        azure_deployment = "gpt-5-mini"

    # Replace these with real values
        custom_model = AzureChatOpenAI(
            model = self.model,
            azure_endpoint = azure_endpoint,
            azure_deployment=azure_deployment,
            openai_api_key = self.api_key,
            openai_api_version = openai_api_version,
        )

        self.azure_openai = AzureOpenAI(model=custom_model)

    def compute(self, prediction_description: str, reference_description: str) -> float:
        """
        Compute several criteria scores between prediction and reference descriptions.
        """
        weight_hallucination = 0.6
        weight_completeness = 0.4
        print("Reference inside metric:", reference_description)

        # Extract run_id if available in prediction data
        if isinstance(prediction_description, list) and len(prediction_description) > 0:
            pred_data = prediction_description[0]
            if 'current_run_id' in pred_data:
                self.run_id = pred_data['current_run_id']

        reference_description = reference_description[0]["expected_description"]
        question = prediction_description[0].get(
            "question", "Answer the question based on the collected facts."
        )

        prediction_description = prediction_description[0]["description"]
        table_name = list(prediction_description.keys())[0]
        print("Table name", table_name)

        prediction_description = prediction_description[table_name]

        print("Prediction inside metric:", prediction_description)
        print("Reference after indexing:", reference_description)
        factual_accuracy = GEval(
            name="Factual Accuracy",
            model = self.azure_openai,
            criteria="Evaluate whether the actual output contains any made-up, incorrect, or fabricated facts when compared to the expected output. Penalize heavily for invented information.",
            evaluation_params=[LLMTestCaseParams.INPUT, 
                               LLMTestCaseParams.ACTUAL_OUTPUT, 
                               LLMTestCaseParams.EXPECTED_OUTPUT],
            threshold=0.7
        )

        completeness = GEval(
            name="Completeness",
            model = self.azure_openai,
            criteria="Evaluate how much of the key information from the expected output is covered in the actual output. Check for missing variables, descriptions, or important details.",
            evaluation_params=[LLMTestCaseParams.INPUT, 
                               LLMTestCaseParams.ACTUAL_OUTPUT, 
                               LLMTestCaseParams.EXPECTED_OUTPUT],
            threshold=0.7
        )
    
        test_case = LLMTestCase(
            input=question,
            #input="Provide a comprehensive description of the MineProcessAssays table, including detailed variable descriptions for GRDFe_A, RCV_PCT, and SMP_RUNID with their data types, purposes, expected ranges, common issues, validation rules, and relationships to other tables.",
            actual_output=prediction_description,
            expected_output=reference_description,
            retrieval_context=[reference_description] 
        )

        factual_accuracy_score, factual_accuracy_reason = self.convergence_geval_loop(factual_accuracy, test_case, n_runs = 20, max_retries = 3, min_std_error=0.05, n_runs_min=5)
        completeness_score, completeness_reason = self.convergence_geval_loop(completeness, test_case, n_runs = 20, max_retries = 3, min_std_error=0.05, n_runs_min=5)

        aggregated_reasoning = self.aggregate_reasons([factual_accuracy_reason, completeness_reason])
        overall_score = (weight_hallucination*factual_accuracy_score + weight_completeness*completeness_score)

        # Store GEval metrics in database if run_id is available
        if self.run_id:
            try:
                db.add_geval_metric(
                    run_id=self.run_id,
                    metric_name="factual_accuracy",
                    score=int(factual_accuracy_score * 10),  # Convert to 0-10 scale
                    reasoning=str(factual_accuracy_reason) if factual_accuracy_reason else None,
                )
                db.add_geval_metric(
                    run_id=self.run_id,
                    metric_name="completeness",
                    score=int(completeness_score * 10),  # Convert to 0-10 scale
                    reasoning=str(completeness_reason) if completeness_reason else None,
                )
                db.add_geval_metric(
                    run_id=self.run_id,
                    metric_name="overall_geval",
                    score=int(overall_score * 10),  # Convert to 0-10 scale
                    reasoning=aggregated_reasoning if aggregated_reasoning else None,
                )
            except Exception as e:
                logger.warning(f"Failed to store GEval metrics in database: {e}")

        return overall_score, aggregated_reasoning

    def convergence_geval_loop(self, metric: BaseMetric, test_case: LLMTestCase, n_runs: int = 10, max_retries: int = 3, n_runs_min: int = 10, min_std_error: float = 0.05):

        retries = 0
        sucessful_runs = 0
        scores = list()
        reasons = list()
        n = 0
        
        while retries < max_retries and sucessful_runs < n_runs:
            try:
                print(f"{metric.name} run {n}")
                score = metric.measure(test_case)
                scores.append(score)

                reasons.append(metric.reason)

                print(f"{metric.name}:", score)
                sucessful_runs += 1
                n += 1

                partial_std_deviation = np.std(scores)
                partial_std_error = partial_std_deviation / np.sqrt(len(scores))
                
                print(f"Standard Error in run {n}: {partial_std_error}")
                print(f"Reason: {metric.reason}")

                if n >= n_runs_min and partial_std_error < min_std_error: # Confidence Interval = 0.95 if min_std_error = 0.05
                    print(f"{metric.name} converged after {n} runs.")
                    return np.mean(np.array(scores)), reasons
                
            except Exception as e:
                print(f"Error during {metric.name} evaluation: {e}. Retrying...")
                retries += 1
                continue

        # If the loop exits without convergence, return the average score and reasons from all runs
        if scores:
            return np.mean(np.array(scores)), reasons
        return 0.0, []

    def aggregate_reasons(self, reasons: List[str]) -> str:

        prompt_aggregation = f"""
        You are an expert AI assistant specialized in summarizing evaluation feedback.
        Given multiple reasoning statements from different evaluation runs, your task is to aggregate them into a single coherent reasoning that captures the key points.  The aggregated reasoning must be as general as possible, rather than using specific names or methods.
        
        Here is a list of reasons for different evaluation metrics:
        {reasons}"""

        reasoning_aggregation = self.azure_openai.generate(prompt_aggregation)

        return reasoning_aggregation

class LLMJudgeMetric(Metric):
    """LLM Judge Metric."""
    
    def __init__(self, name: str = "llm_judge"):
        super().__init__(name)
       
    
    def compute(self, predictions: List[Any], references: List[Any]) -> float:
        """Compute exact match score."""
        
        logger.debug(f"Predictions: {predictions}")
        logger.debug(f"References: {references}")

        scores = []
        for pred, ref in zip(predictions, references):
            # Here we would call an LLM to judge the quality of pred against ref
            # For simplicity, we'll use a dummy score
            judge_score = random.uniform(0, 10)  # Dummy score between 0 and 10
            scores.append(judge_score)

        logger.debug(f"Scores: {scores}")
        logger.debug(f"Mean score: {np.mean(np.array(scores))}")
        return np.mean(np.array(scores))
    

def evaluate_prompt_kontex(prompts:dict, dataset: dict):

    descriptions_dataset = list()
    scores_dataset = list()
    for datapoint in dataset:
        description = run_conversation_simulation(
            initial_prompts=prompts,
            simulated_users=datapoint["users_with_knowledge"],
            full_knowledge=datapoint["full_knowledge"],
            seed=42,
        )
    
        descriptions_dataset.append(description)

        similarity_matrix, score = compute_similarity(description, datapoint)

        scores_dataset.append(score)
    return descriptions_dataset, scores_dataset

def compute_similarity(description, datapoint):
    """
    Compute the semantic similarity between the description and the facts in full_knowledge.
    """

    desc_texts = list(description.values())
    domain = list(datapoint["full_knowledge"].domains.keys())[0]
    facts_texts = list(datapoint["full_knowledge"].domains[domain].facts.values())

    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Compute embeddings
    desc_emb = model.encode(desc_texts, convert_to_tensor=True)
    facts_emb = model.encode(facts_texts, convert_to_tensor=True)

    # Cosine simlarity
    similarity_matrix = util.cos_sim(desc_emb, facts_emb)

    # Aggregate the scores (e.g., mean of max similarities for each description)
    score = similarity_matrix.max(dim=1).values.mean().item()

    return similarity_matrix, score

def generate_pareto_dataset(seed = 42):
    from collections import defaultdict
    table_themes = ["mining", "healthcare"] #"finance", "technology", "retail"]#, "education"]
    
    dataset = list()

    for theme in table_themes:
        config = EDDRunConfig(
                max_hier_depth=2,
                n_employees=5,
                mean_degree=math.ceil(5 ** (1 / 2)),
                alpha=0.1,
                decay=0.8,
                forgetting_chance=0.7,
                n_patients_zero=1,
                connections=1.5,
                table_info=[(theme, 2, 0.8)],
            )

        run, simulated_users, full_knowledge = edd_simulation(config, seed, theme = theme)
    
        domain_name = list(full_knowledge.domains.keys())[0]
        print("DOMAIN NAME", domain_name)
        domain_description = full_knowledge.domains[domain_name].description
        column_descriptions = full_knowledge.domains[domain_name].facts

        theme_dict = dict()
        theme_dict["full_knowledge"] = full_knowledge
        theme_dict["run_id"] = run.id
        theme_dict["users_with_knowledge"] = simulated_users
        theme_dict["question"] = f"Describe the dataset related to {theme} operations, including key attributes and their significance."
        theme_dict["expected"] = 10

        print(domain_description)
        print(column_descriptions)
        column_keys = column_descriptions.keys()
        column_descriptions = "\n".join([f"- {col}: {column_descriptions[col]}" for col in column_keys])
        theme_dict["expected_description"] = domain_name + "\n" + domain_description + "\n" + column_descriptions

        logger.debug(f"Table description: {domain_description}")
        logger.debug(f"Column descriptions: {column_descriptions}")
        logger.debug(f"Simulated users: {simulated_users}")
        dataset.append(theme_dict)

    return dataset

def generate_hotpot_pareto_dataset(seed = 42):

    with open("../data/hotpot_train_v1.1.json") as f:
        data = json.load(f)

    qa = [
            [item["question"], item["context"], item["answer"]] for item in data if item["level"] == "hard"
        ]
    
    dataset = list()

    for i in range(3):
        theme = f"hotpotqa_question_{i}"

        external_question = qa[i][0]

        raw_data = qa[i][1]

        pre_existing_data = (raw_data, "HotPotQA", theme)

        config = EDDRunConfig(
            max_hier_depth=10,
            n_employees=10,
            mean_degree=math.ceil(5 ** (1 / 2)),
            # alpha=0.1, original do kontex
            alpha=0,  # para não haver vazamento de informação entre os especialistas
            decay=0.8,
            # forgetting_chance=0.7, original do kontex
            forgetting_chance=0,  # para não haver esquecimento dos especialistas com relação ao que sabem
            n_patients_zero=1,
            connections=1.5,
            table_info=[("mining", 3, 0.8)],
            pre_existing_data=pre_existing_data,
            external_specialist_role=True,
            single_knowledge_employee=True,
            )

        run, simulated_users, full_knowledge = edd_simulation(config, seed, external_data=True)
        
        domain_name = list(full_knowledge.domains.keys())[0]
        print("DOMAIN NAME", domain_name)
        domain_description = full_knowledge.domains[domain_name].description
        column_descriptions = full_knowledge.domains[domain_name].facts

        theme_dict = dict()
        theme_dict["full_knowledge"] = full_knowledge
        theme_dict["run_id"] = run.id
        theme_dict["users_with_knowledge"] = simulated_users
        theme_dict["question"] = external_question
        theme_dict["expected"] = 10

        print(domain_description)
        print(column_descriptions)
        column_keys = column_descriptions.keys()
        column_descriptions = "\n".join([f"- {col}: {column_descriptions[col]}" for col in column_keys])
        theme_dict["expected_description"] = domain_name + "\n" + domain_description + "\n" + column_descriptions

        logger.debug(f"Table description: {domain_description}")
        logger.debug(f"Column descriptions: {column_descriptions}")
        logger.debug(f"Simulated users: {simulated_users}")
        dataset.append(theme_dict)

    return dataset
    # metric to be used to evolve GEPA will be numeric, by comparing the expected final score with the one the critic uses
    # if the difference between the two decreased from previous gepa iteration, then the prompt is better
    # TODO: need to include the similarity metric in the final_score calculation, becaususe the critic only 
    # evaluates how the answer is written and not so much its content

    # TODO: Use reasoning of deepeval metrics to create final_score 

def get_data_from_df(df: pd.DataFrame, n: int = 5) -> FullKnowledge:
    dataset = list()
    i=0
    for row in df.itertuples():
        table_name = row.table_name
        table_description = row.description
        facts = eval(row.facts)
        domain_name= row.table_theme    
        domain_description = table_description
        column_descriptions = facts
        full_knowledge = FullKnowledge(title=table_name)
        full_knowledge.add_domain(table_name, table_description)

        for col, desc in facts.items():
            full_knowledge.add_fact(table_name, col, desc)

        print("DOMAIN NAME", domain_name)
        print(facts)
        domain_description = full_knowledge.domains[table_name].description
        column_descriptions = full_knowledge.domains[table_name].facts

        theme_dict = dict()
        theme_dict["full_knowledge"] = full_knowledge
        theme_dict["question"] = f"Describe the dataset related to {domain_name} operations, including key attributes and their significance."
        theme_dict["expected"] = 10

        column_keys = column_descriptions.keys()
        column_descriptions = "\n".join([f"- {col}: {column_descriptions[col]}" for col in column_keys])
        theme_dict["expected_description"] = domain_name + "\n" + domain_description + "\n" + column_descriptions

        dataset.append(theme_dict)
        i+=1
        if i >= n:
            break
    return dataset



In [7]:
async def main():

    dataset = generate_hotpot_pareto_dataset()
    display(dataset)
    
    pareto_size = 1
    feedback_size = 1
    
    print('dataset lido com sucesso...')

    dataset = dataset[:pareto_size + feedback_size]

    # 2. System with 2 modules: questioner and critique
    system = CompoundAISystem(
        modules={
            "description_building": LanguageModule(
                id="hotpot_description_builder",
                prompt= hotpotqa_description_building_role,
                model_weights="gpt-5-mini"
            )
        },
        control_flow=KontexFlowGeneralized(),
        input_schema=IOSchema(
            fields={"full_knowledge": FullKnowledge},
            required=["full_knowledge"]
        ),
        output_schema=IOSchema(
            fields={"output": int},
            required=["output"]
        ),
        system_id="kontex"
    )

    print('KontexFlow finalizado...')

    print('inicianto EnvConfig...')

    config = EnvConfig(env_file = "../.env")
    # Check for API key
    api_key = config.api_key
    base_url = config.base_url
    # print("api key:", api_key)
    if not api_key:
        print("Please set OPENAI_API_KEY environment variable")
        print("   export OPENAI_API_KEY='your-api-key-here'")
        return
    
    print("Found API key")
    # 3. Configuration
    config = GEPAConfig(
        inference=InferenceConfig(
            provider="openai",
            model="gpt-5-mini",
            api_key=api_key,
            max_tokens=4096,
            temperature=0.1,
            timeout=30,
            base_url=base_url,
            retry_attempts=3
        ),
        optimization=OptimizationConfig(
            budget=20,
            pareto_set_size=pareto_size,  # change pareto set size
            minibatch_size=feedback_size,
            enable_crossover=True,
            crossover_probability=0.3,
            mutation_types=["rewrite", "insert"]
        ),
        database=DatabaseConfig(
            url="sqlite:///gepa_quickstart.db"
        ),
        observability=ObservabilityConfig(
            log_level="INFO",
            log_file="gepa_quickstart.log",
            enable_logging=True
        )
    )

    # 4. Create evaluator (need to change metrics for Kontex)
    evaluator = SimpleFeedbackEvaluator([
        # AverageDiffScore(name="average_score")
        GEvalMetric(name="geval_metric")
    ])
    
    # 5. Create inference client
    print(config.inference.provider)
    inference_client = InferenceFactory.create_client(config.inference)

    # 6. Create optimizer and run optimization
    print("🔄 Starting optimization...")
    print(f"   Budget: {config.optimization.budget} rollouts")
    print(f"   Dataset size: {len(dataset)} examples")
    print()
    
    optimizer = GEPAOptimizer(
        config=config,
        evaluator=evaluator,
        inference_client=inference_client
    )
    
    try:
        result = await optimizer.optimize(system, dataset, max_generations=5)
        
        # 7. Display results
        print("✅ Optimization completed!")
        print("=" * 50)
        print(f"🎯 Best score: {result.best_score:.3f}")
        print(f"🔄 Total rollouts: {result.total_rollouts}")
        print(f"💰 Total cost: ${result.total_cost:.4f}")
        print(f"📊 Pareto frontier size: {result.pareto_frontier.size()}")
        print()
        


        # Show the optimized prompt
        best_description_builder_module = result.best_system.modules["description_builder"]
        # best_critique_module = result.best_system.modules["critique"]
        print("🧠 Optimized description_building prompt:")
        print("-" * 30)
        print(best_description_builder_module.prompt)

        logger.info(f"Best description_building prompt: \n {best_description_builder_module.prompt}")


        # # Show the optimized prompt
        # best_questioning_module = result.best_system.modules["questioning"]
        # # best_critique_module = result.best_system.modules["critique"]
        # print("🧠 Optimized questioning prompt:")
        # print("-" * 30)
        # print(best_questioning_module.prompt)

        # logger.info(f"Best questioning prompt: \n {best_questioning_module.prompt}")



        # logger.info(f"Best critique prompt: \n {best_critique_module.prompt}")
        # print("🧠 Optimized critique prompt:")
        # print("-" * 30)
        # # print(best_critique_module.prompt)
        # print("-" * 30)
        # print()
        
        # # Test the optimized system
        # print("🧪 Testing optimized system...")
        # test_examples = [
        #     "This movie was absolutely incredible!",
        #     "I'm disappointed with this purchase.",
        #     "The weather is fine today."
        # ]
        
        # for test_text in test_examples:
        #     try:
        #         # Simulate running the optimized system
        #         input_data = {"text": test_text}
        #         # In a real scenario, you'd run: result = await result.best_system.execute(input_data, inference_client)
        #         # For demo, we'll just show the input
        #         print(f"   Input: '{test_text}'")
        #         print(f"   System: sentiment_classifier")
        #         print()
        #     except Exception as e:
        #         print(f"   Error testing: {e}")
        
        # Show optimization statistics
        stats = optimizer.get_statistics()
        print("📊 Optimization Statistics:")
        print(f"   Generations completed: {stats.get('generations', 0)}")
        print(f"   Successful mutations: {stats.get('successful_mutations', 0)}")
        print(f"   Average score improvement: {stats.get('average_improvement', 0):.3f}")
        
    except Exception as e:
        import traceback
        print(f"❌ Optimization failed: {e}")
        print(f"Traceback: {traceback.format_exc()}")
        print("This might be due to API limits or network issues.")
        print("Try again with a smaller budget or check your API key.")
    
    finally:
        # Clean up
        await inference_client.close() if hasattr(inference_client, 'close') else None
        print("\n🎉 Quickstart example completed!")

    # prompts = {
    #     "questioner_prompt": questioner_prompt,
    #     "critique_prompt": critique_prompt
    # }

    # descriptions_dataset, scores_dataset = evaluate_prompt_kontex(prompts, dpareto)

In [8]:
if __name__ == "__main__":
    await main()


2026-03-20 15:24:31.522 | INFO     | kontex.simulation.edd.simulation:edd_simulation:46 - Starting simulation for knowledge distribution in a company...
2026-03-20 15:24:31.523 | INFO     | kontex.simulation.edd.general_knowledge:generate:217 - Transforming data for dataset: HotPotQA with domain: hotpotqa_question_0
2026-03-20 15:24:31.523 | DEBUG    | kontex.simulation.edd.general_knowledge:generate:233 - Transformed knowledge: title='HotPotQA' domains={'hotpotqa_question_0': DomainKnowledge(title='hotpotqa_question_0', description='A collection of facts for the hotpotqa_question_0 domain.', facts={'Lisa Simpson': 'Lisa Marie Simpson is a fictional character in the animated television series "The Simpsons".  She is the middle child and most intelligent of the Simpson family.  Voiced by Yeardley Smith, Lisa first appeared on television in "The Tracey Ullman Show" short "Good Night" on April 19, 1987.  Cartoonist Matt Groening created and designed her while waiting to meet James L. Broo

DOMAIN NAME hotpotqa_question_0
A collection of facts for the hotpotqa_question_0 domain.
{'Lisa Simpson': 'Lisa Marie Simpson is a fictional character in the animated television series "The Simpsons".  She is the middle child and most intelligent of the Simpson family.  Voiced by Yeardley Smith, Lisa first appeared on television in "The Tracey Ullman Show" short "Good Night" on April 19, 1987.  Cartoonist Matt Groening created and designed her while waiting to meet James L. Brooks.  Groening had been invited to pitch a series of shorts based on his comic "Life in Hell", but instead decided to create a new set of characters.  He named the elder Simpson daughter after his younger sister Lisa Groening.  After appearing on "The Tracey Ullman Show" for three years, the Simpson family were moved to their own series on Fox, which debuted on December 17, 1989.', 'Marge Simpson': 'Marjorie Jacqueline "Marge" Simpson (née Bouvier) is a fictional character in the American animated sitcom "The Si

2026-03-20 15:24:48.151 | INFO     | kontex.llm.model:chat_request:124 - **Usage Success** | Prompt_tokens: 149 | Output_tokens: 695 | Total_tokens: 844
2026-03-20 15:24:48.223 | DEBUG    | kontex.llm.agents.base:answer:114 - Adding response to history in user
2026-03-20 15:24:48.224 | INFO     | kontex.simulation.edd.company_structure:generate_hierarchy_levels:59 - Generating hierarchy levels with max_hier_depth=10, mean_degree=3
2026-03-20 15:24:48.225 | INFO     | kontex.simulation.edd.company_spread_knowledge:spread_knowledge_with_neighbors:62 - Starting knowledge spread from ['Wei Chen'] with decay=0.8, alpha=0.0, max_steps=10, forgetting_chance=0.0, single_knowledge_employee=True
2026-03-20 15:24:48.225 | INFO     | kontex.simulation.edd.company_spread_knowledge:spread_knowledge_with_neighbors:77 - Initializing knowledge: Unique piece per node (External Data mode).
2026-03-20 15:24:48.226 | INFO     | kontex.simulation.edd.company_employees:create_company_employees:102 - Selectin

DOMAIN NAME hotpotqa_question_1
A collection of facts for the hotpotqa_question_1 domain.
{'India': 'India, officially the Republic of India ("Bhārat Gaṇarājya"), is a country in South Asia.  It is the seventh-largest country by area, the second-most populous country (with over 1.2 billion people), and the most populous democracy in the world.  It is bounded by the Indian Ocean on the south, the Arabian Sea on the southwest, and the Bay of Bengal on the southeast.  It shares land borders with Pakistan to the west; China, Nepal, and Bhutan to the northeast; and Myanmar (Burma) and Bangladesh to the east.  In the Indian Ocean, India is in the vicinity of Sri Lanka and the Maldives.  India\'s Andaman and Nicobar Islands share a maritime border with Thailand and Indonesia.', 'List of companies of India': 'India is a country in South Asia.  It is the seventh-largest country by area, the second-most populous country (with over 1.2 billion people), and the most populous democracy in the world

2026-03-20 15:24:54.805 | INFO     | kontex.llm.model:chat_request:124 - **Usage Success** | Prompt_tokens: 149 | Output_tokens: 374 | Total_tokens: 523
2026-03-20 15:24:54.819 | DEBUG    | kontex.llm.agents.base:answer:114 - Adding response to history in user
2026-03-20 15:24:54.820 | INFO     | kontex.simulation.edd.company_structure:generate_hierarchy_levels:59 - Generating hierarchy levels with max_hier_depth=10, mean_degree=3
2026-03-20 15:24:54.822 | INFO     | kontex.simulation.edd.company_spread_knowledge:spread_knowledge_with_neighbors:62 - Starting knowledge spread from ['Wei Chen'] with decay=0.8, alpha=0.0, max_steps=10, forgetting_chance=0.0, single_knowledge_employee=True
2026-03-20 15:24:54.823 | INFO     | kontex.simulation.edd.company_spread_knowledge:spread_knowledge_with_neighbors:77 - Initializing knowledge: Unique piece per node (External Data mode).
2026-03-20 15:24:54.825 | INFO     | kontex.simulation.edd.company_employees:create_company_employees:102 - Selectin

DOMAIN NAME hotpotqa_question_2
A collection of facts for the hotpotqa_question_2 domain.
{'Mount Panorama Circuit': 'Mount Panorama Circuit is a motor racing track located in Bathurst, New South Wales, Australia.  It is situated on a hill with the dual official names of Mount Panorama and Wahluu and is best known as the home of the Bathurst 1000 motor race held each October, and the Bathurst 12 Hour event held each February.  The 6.213 km long track is technically a street circuit, and is a public road, with normal speed restrictions, when no racing events are being run, and there are many residences which can only be accessed from the circuit.', '2016 Intercontinental GT Challenge': 'The 2016 Intercontinental GT Challenge was the first season of the Intercontinental GT Challenge.  The season featured three rounds\xa0— after the cancellation of the 6 Hours of the Americas - starting with Liqui Moly Bathurst 12 Hour on 7 February and the season concluded with the Sepang 12 Hours on 10 

[{'full_knowledge': FullKnowledge(title='HotPotQA', domains={'hotpotqa_question_0': DomainKnowledge(title='hotpotqa_question_0', description='A collection of facts for the hotpotqa_question_0 domain.', facts={'Lisa Simpson': 'Lisa Marie Simpson is a fictional character in the animated television series "The Simpsons".  She is the middle child and most intelligent of the Simpson family.  Voiced by Yeardley Smith, Lisa first appeared on television in "The Tracey Ullman Show" short "Good Night" on April 19, 1987.  Cartoonist Matt Groening created and designed her while waiting to meet James L. Brooks.  Groening had been invited to pitch a series of shorts based on his comic "Life in Hell", but instead decided to create a new set of characters.  He named the elder Simpson daughter after his younger sister Lisa Groening.  After appearing on "The Tracey Ullman Show" for three years, the Simpson family were moved to their own series on Fox, which debuted on December 17, 1989.', 'Marge Simpson

2026-03-20 15:24:55.121 | INFO     | gepa.core.optimizer:info:54 - Starting GEPA optimization system_id=kontex dataset_size=2 budget=20
2026-03-20 15:24:55.121 | ERROR    | gepa.core.optimizer:error:66 - System execution failed error='dict' object has no attribute 'prompt'
2026-03-20 15:24:55.122 | DEBUG    | gepa.core.optimizer:debug:58 - Traceback (most recent call last):
  File "/home/kunumi/projetos/gepa/src/gepa/core/optimizer.py", line 377, in _execute_system_with_tracking
    output_data = await system.execute(data_point, self.inference_client)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kunumi/projetos/gepa/src/gepa/core/system.py", line 171, in execute
    result = await self.control_flow.execute(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_6943/2394716416.py", line 111, in execute
    "description_building": modules.prompt['description_building'],
                            ^^^^^^^^^^^^^^
AttributeError: 'd

dataset lido com sucesso...
KontexFlow finalizado...
inicianto EnvConfig...
✓ Loaded environment variables from ../.env
Found API key
⚠ Warning: .env file not found
openai
🔄 Starting optimization...
   Budget: 20 rollouts
   Dataset size: 2 examples

Trajectories:  [Trajectory(system_id='kontex', input_data={'full_knowledge': FullKnowledge(title='HotPotQA', domains={'hotpotqa_question_1': DomainKnowledge(title='hotpotqa_question_1', description='A collection of facts for the hotpotqa_question_1 domain.', facts={'India': 'India, officially the Republic of India ("Bhārat Gaṇarājya"), is a country in South Asia.  It is the seventh-largest country by area, the second-most populous country (with over 1.2 billion people), and the most populous democracy in the world.  It is bounded by the Indian Ocean on the south, the Arabian Sea on the southwest, and the Bay of Bengal on the southeast.  It shares land borders with Pakistan to the west; China, Nepal, and Bhutan to the northeast; and Myanmar